# 1D CNN-A — Temporal Patterns: When Do Market Regimes Appear?

Assigns every clean window a K-Means cluster label and plots those labels
across time — by hour of day, by day of week, and across the full date range —
to reveal intraday and seasonal structure in the discovered market regimes.

> **Prerequisite:** run `1dcnn_train.ipynb` first — it saves `model.pt` to
> `DATA_DIR / SYMBOL /`.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

from datetime import date

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
csv_path = os.path.join(DATA_DIR, SYMBOL, f"{TIMEFRAME}.csv")
df = pd.read_csv(csv_path, parse_dates=["timestamp"], nrows=MAX_BARS)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df):,} bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
max_bars_display = 'all' if MAX_BARS is None else f'{MAX_BARS:,}'
print(f"(MAX_BARS={max_bars_display})")
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
df = df.drop_duplicates(subset=["timestamp"])
df = df.dropna()

df.isnull().sum()
df.duplicated(subset=["timestamp"]).sum()
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# EMAs
df["ema_9"] = df["close"].ewm(span=9, adjust=False).mean()
df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

# MACD components
df["macd_12"] = df["close"].ewm(span=12, adjust=False).mean()
df["macd_26"] = df["close"].ewm(span=26, adjust=False).mean()

# MACD line
df["macd"] = df["macd_12"] - df["macd_26"]

# MACD signal line (9 EMA of MACD)
df["macd_9"] = df["macd"].ewm(span=9, adjust=False).mean()

# MACD histogram (optional but commonly used)
df["macd_hist"] = df["macd"] - df["macd_9"]

# Candle details
df["body"] = df["close"] - df["open"]
df["upper_wick"] = df["high"] - df[["open", "close"]].max(axis=1)
df["lower_wick"] = df[["open", "close"]].min(axis=1) - df["low"]



# other
df["return"] = df["close"].pct_change()
df["vol_return"] = df["volume"].pct_change()
df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["volume_ratio"] = (
    df["volume"] /
    df["volume"].rolling(20).mean()
)

# display sample of new features
# df[["timestamp", "close", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

df[df["body"] != 0][
    [
        "timestamp",
        "close",
        "ema_9",
        "ema_21",
        "ema_50",
        "macd",
        "macd_9",
        "macd_hist",
        "body",
        "upper_wick",
        "lower_wick",
        "return",
        "vol_return",
        "log_return",
        "volume_ratio",
    ]
].tail()

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
df = df.dropna().reset_index(drop=True)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

# feature_cols is defined in config.py — edit it there to change which features are used
# scaler = StandardScaler()
# df[feature_cols] = scaler.fit_transform(df[feature_cols])

scaler = RobustScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
# Often better for financial data due to outliers

df[["timestamp", "open", "high", "low", "close", "volume", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
n_features = len(feature_cols)
print(f"Features ({n_features}):", feature_cols)
print("Data shape:", df[feature_cols].shape)

data = df[feature_cols].to_numpy(dtype=np.float32)

X_raw = np.lib.stride_tricks.sliding_window_view(
    data,
    window_shape=WINDOW_SIZE,
    axis=0
).transpose(0, 2, 1)   # → (N, WINDOW_SIZE, n_features)

print("X_raw shape:", X_raw.shape)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
diffs_sec = df["timestamp"].diff().dt.total_seconds().fillna(0).to_numpy()
gap_positions = np.where(diffs_sec > 300)[0]   # > 5 min between consecutive bars

valid_mask = np.ones(len(X_raw), dtype=bool)
for gp in gap_positions:
    lo = max(0, gp - WINDOW_SIZE + 1)
    hi = min(len(X_raw), gp + 1)
    valid_mask[lo:hi] = False

X_clean = X_raw[valid_mask]
print(f"Gap positions: {len(gap_positions)}")
print(f"Removed {(~valid_mask).sum():,} gap-spanning windows")
print(f"Clean windows: {X_clean.shape[0]:,}  shape: {X_clean.shape}")

## 13. Autoencoder Model
Encoder compresses `(batch, 14, 64)` → latent vector `(batch, LATENT_DIM)`.
Decoder reconstructs `(batch, 14, 64)` from the latent vector.
Training loss is reconstruction MSE — no labels needed.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 32, 32)
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 64, 16)
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 128, 8)
        )
        self.fc = nn.Linear(128 * 8, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc(h)


class Decoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 8)
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=2, padding=1),   # → (batch, 64, 16)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),    # → (batch, 32, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(32, n_features, kernel_size=4, stride=2, padding=1),  # → (batch, 14, 64)
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 128, 8)
        return self.deconv(h)


class ConvAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.encoder = Encoder(n_features, latent_dim)
        self.decoder = Decoder(n_features, latent_dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

In [ ]:
# WindowDataset is needed by the latent-extraction DataLoader (Section 15)
class WindowDataset(Dataset):
    def __init__(self, X):
        self.X = X
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]   # input == reconstruction target


In [ ]:
model_path = os.path.join(DATA_DIR, SYMBOL, "model.pt")

model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()

print(f"Loaded  : {model_path}")
print(f"  Architecture  : ConvAutoencoder(n_features={n_features}, latent_dim={LATENT_DIM})")
print(f"  Input shape   : (batch, {n_features}, {WINDOW_SIZE})  — channels-first")
print(f"  Device        : {DEVICE}")
print(f"  Parameters    : {sum(p.numel() for p in model.parameters()):,}")

## 15. Extract Latent Vectors & Assign Cluster Labels

In [ ]:
from sklearn.cluster import KMeans

all_loader = DataLoader(WindowDataset(
    torch.tensor(X_clean).permute(0, 2, 1)
), batch_size=BATCH_SIZE, shuffle=False)

model.eval()
Z_list = []
with torch.no_grad():
    for batch in all_loader:
        Z_list.append(model.encoder(batch.to(DEVICE)).cpu().numpy())

Z = np.concatenate(Z_list)   # (N_clean, LATENT_DIM)
print(f'Latent matrix Z: {Z.shape}')

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
labels = kmeans.fit_predict(Z)   # (N_clean,)
print('Cluster sizes:', dict(zip(*np.unique(labels, return_counts=True))))

## 18. Attach Timestamps to Clean Windows

In [ ]:
# Each window ends at bar (i + WINDOW_SIZE - 1); we tag it with that bar's timestamp.
# valid_mask (built in gap-filter cell) maps clean window index → original bar index.
valid_indices = np.where(valid_mask)[0]          # original bar positions of clean windows
window_end_idx = valid_indices + WINDOW_SIZE - 1 # last bar of each window
window_end_idx = np.clip(window_end_idx, 0, len(df) - 1)

timestamps = pd.to_datetime(df['timestamp'].iloc[window_end_idx].values)

regime_df = pd.DataFrame({
    'timestamp': timestamps,
    'cluster':   labels,
})
regime_df['hour']    = regime_df['timestamp'].dt.hour + regime_df['timestamp'].dt.minute / 60
regime_df['weekday'] = regime_df['timestamp'].dt.day_name()
regime_df['date']    = regime_df['timestamp'].dt.normalize()

print(f'regime_df: {len(regime_df):,} rows  |  date range: '
      f'{regime_df["timestamp"].min().date()} → {regime_df["timestamp"].max().date()}')

## 19. Full-Timeline Scatter

Each dot is one 64-bar window, coloured by its cluster. Vertical bands of colour
indicate sustained market regimes; rapid colour switching = volatile transitions.

In [ ]:
import matplotlib.dates as mdates

cmap = plt.get_cmap('tab10')

fig, ax = plt.subplots(figsize=(20, 4))
for k in range(N_CLUSTERS):
    mask = regime_df['cluster'] == k
    ax.scatter(regime_df.loc[mask, 'timestamp'], [k] * mask.sum(),
               c=[cmap(k)], s=1, alpha=0.4, label=f'Cluster {k}')

ax.set_yticks(range(N_CLUSTERS))
ax.set_yticklabels([f'Cluster {k}' for k in range(N_CLUSTERS)], fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_title(f'{SYMBOL} {TIMEFRAME} — Cluster Label Over Full Timeline', fontsize=13)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

## 20. Hour-of-Day Heatmap

What time of day does each market regime tend to appear?
Bright cells = that cluster frequently occurs at that hour.

In [ ]:
# Count occurrences per cluster per hour-bin (30-min bins)
regime_df['hour_bin'] = (regime_df['hour'] * 2).astype(int) / 2  # round to 0.5h
heatmap_data = (
    regime_df.groupby(['hour_bin', 'cluster'])
    .size()
    .unstack(fill_value=0)
)
# Normalise each hour row so colours show proportion rather than raw count
heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heatmap_norm.T, aspect='auto', cmap='YlOrRd', vmin=0)
ax.set_xticks(range(len(heatmap_norm.index)))
ax.set_xticklabels(
    [f'{int(h):02d}:{int((h % 1) * 60):02d}' for h in heatmap_norm.index],
    rotation=45, ha='right', fontsize=8
)
ax.set_yticks(range(N_CLUSTERS))
ax.set_yticklabels([f'Cluster {k}' for k in range(N_CLUSTERS)], fontsize=9)
ax.set_xlabel('Hour of Day (ET)')
ax.set_title(f'{SYMBOL} — Cluster Frequency by Hour of Day', fontsize=12)
plt.colorbar(im, ax=ax, label='Proportion of windows at that hour')
plt.tight_layout()
plt.show()

## 21. Day-of-Week Distribution

Does any cluster dominate a particular weekday?
Seasonal trading patterns (Monday gaps, Friday de-risking) often show up here.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
day_data = (
    regime_df.groupby(['weekday', 'cluster'])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)
day_norm = day_data.div(day_data.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
bottom = np.zeros(len(day_order))
for k in range(N_CLUSTERS):
    vals = day_norm[k].values if k in day_norm.columns else np.zeros(len(day_order))
    ax.bar(day_order, vals, bottom=bottom, color=cmap(k), label=f'Cluster {k}', width=0.7)
    bottom += vals

ax.set_ylabel('Proportion of windows')
ax.set_title(f'{SYMBOL} — Cluster Frequency by Day of Week', fontsize=12)
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 22. What You're Seeing

### Full-Timeline Scatter
Long runs of the same colour = sustained market regimes (e.g. a trending week, earnings volatility).
Rapid colour switching = choppy, transitional markets.

### Hour-of-Day Heatmap
Common patterns in TSLA 1-minute data:
- **Open volatility cluster** (9:30–10:00 ET) — high `volume_ratio`, wide candles
- **Lunchtime lull cluster** (12:00–13:00 ET) — flat `macd`, low `volume_ratio`
- **Close cluster** (15:30–16:00 ET) — rising volume, directional momentum

### Day-of-Week
Monday and Friday often have distinct distributions due to gap risk and
position-squaring. If one cluster is Monday-heavy, inspect its centroids
in `latent_cluster.ipynb` for gap-open characteristics.

**Next step:** take any cluster that stands out here and look at its centroid
line plot in `latent_cluster.ipynb` to understand what the pattern actually looks like.